# hftbacktest - TW Stock 

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.tw_stock_data_to_npz import convert_tw_stock_to_npz
from scripts.tw_stock_hftbacktest import (
    BacktestConfig,
    build_backtest,
    close_backtest,
    import_hftbacktest,
    state_snapshot,
    submit_limit_order,
    wait_for_bbo,
)


* Set `SYMBOL`, `START_DATE`, `END_DATE`, `START_TIME`, and `END_TIME` in the next cell. 
* The notebook calls `scripts.tw_stock_data_to_npz.convert_tw_stock_to_npz()` to generate the npz data file before running the backtest.

In [ ]:
SYMBOL = "2308"
START_DATE = "2026-01-12"
END_DATE = "2026-01-15"
START_TIME = "12:00:00"
END_TIME = "13:30:00"

DATA_FILE, event_data = convert_tw_stock_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
)

In [ ]:

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(data=DATA_FILE, order_latency_ns=0)
DATA_FILE

## Strategy 1: ask/buy @ bbo

In [ ]:
def ns_to_datetime(ns):
    return pd.to_datetime(ns, unit="ns", utc=True).tz_convert("Asia/Taipei") if pd.notna(ns) else pd.NaT


def get_order(hbt, asset_no, order_id):
    if order_id is None:
        return None
    try:
        return hbt.orders(asset_no).get(order_id)
    except Exception:
        return None


def event_local_ts_for_exch_ts(exch_ts, fallback=pd.NA):
    if pd.isna(exch_ts):
        return fallback
    matches = event_data["exch_ts"] == int(exch_ts)
    if matches.any():
        return int(event_data["local_ts"][matches][0])
    return fallback


def record_order_state(hbt, hbtpkg, asset_no, config, label, strategy, queue_model=None, round_no=None,
                       side=None, order_id=None, px=None, qty=None, rc=None, response=None, step=None,
                       is_fill=False):
    order = get_order(hbt, asset_no, order_id)
    current_ts = int(hbt.current_timestamp)
    exch_ts = fill_ts = order_exch_ts = order_local_ts = pd.NA
    local_ts = current_ts
    order_status = exec_qty = exec_price = leaves_qty = pd.NA
    if order is not None:
        order_exch_ts = int(order.exch_timestamp)
        order_local_ts = int(order.local_timestamp)
        exch_ts = order_exch_ts
        order_status = int(order.status)
        exec_qty = float(order.exec_qty)
        exec_price = float(order.exec_price)
        leaves_qty = float(order.leaves_qty)
        if is_fill:
            local_ts = event_local_ts_for_exch_ts(order_exch_ts, fallback=current_ts)
            fill_ts = local_ts

    row = {
        "strategy": strategy,
        "queue_model": queue_model,
        "label": label,
        "round": round_no,
        "step": step,
        "side": side,
        "order_id": order_id,
        "price": px,
        "qty": qty,
        "rc": rc,
        "response": response,
        "current_ts": current_ts,
        "exch_ts": exch_ts,
        "local_ts": local_ts,
        "fill_ts": fill_ts,
        "order_exch_ts": order_exch_ts,
        "order_local_ts": order_local_ts,
        "current_time": ns_to_datetime(current_ts),
        "exch_time": ns_to_datetime(exch_ts),
        "local_time": ns_to_datetime(local_ts),
        "fill_time": ns_to_datetime(fill_ts),
        "order_exch_time": ns_to_datetime(order_exch_ts),
        "order_local_time": ns_to_datetime(order_local_ts),
        "order_status": order_status,
        "exec_qty": exec_qty,
        "exec_price": exec_price,
        "leaves_qty": leaves_qty,
    }
    row.update(state_snapshot(hbt, asset_no, config.contract_size))
    return row


def run_aggressive_fill_strategy(hbt, hbtpkg, config=CONFIG, qty=1.0, round_trips=1, response_timeout_ns=10_000_000):
    asset_no = 0
    rows = []

    def record(label, round_no=None, side=None, order_id=None, px=None, rc=None, response=None, is_fill=False):
        rows.append(record_order_state(
            hbt, hbtpkg, asset_no, config, label, "aggressive_bbo", config.queue_model,
            round_no=round_no, side=side, order_id=order_id, px=px,
            qty=qty if side is not None else None, rc=rc, response=response, is_fill=is_fill,
        ))

    wait_for_bbo(hbt, asset_no)
    record("initial_bbo")

    order_id = 10_001
    for round_no in range(1, round_trips + 1):
        depth = hbt.depth(asset_no)
        px = float(depth.best_ask)
        record("before_buy", round_no, "buy", order_id, px)
        rc = submit_limit_order(hbt, hbtpkg, asset_no, order_id, "buy", px, qty)
        response = hbt.wait_order_response(asset_no, order_id, response_timeout_ns)
        record("after_buy", round_no, "buy", order_id, px, rc, response, is_fill=True)
        hbt.clear_inactive_orders(asset_no)
        order_id += 1

        depth = hbt.depth(asset_no)
        px = float(depth.best_bid)
        record("before_sell", round_no, "sell", order_id, px)
        rc = submit_limit_order(hbt, hbtpkg, asset_no, order_id, "sell", px, qty)
        response = hbt.wait_order_response(asset_no, order_id, response_timeout_ns)
        record("after_sell", round_no, "sell", order_id, px, rc, response, is_fill=True)
        hbt.clear_inactive_orders(asset_no)
        order_id += 1

    record("final_state")
    return pd.DataFrame(rows)

In [ ]:
hbt = build_backtest(CONFIG, hbtpkg)
try:
    strategy_output = run_aggressive_fill_strategy(hbt, hbtpkg, CONFIG, qty=1.0, round_trips=1)
finally:
    close_backtest(hbt)

strategy_output

## Strategy 2: ask/bid @ ask1/bid1 (Comparing Queue Model)

* submit passive buy at bid1 and passive sell at ask1
* compare `risk_adverse` and `log_prob` queue models by fill timestamp
* record exchange time, local time, and fill time

In [ ]:
QUEUE_MODELS = ["risk_adverse", "log_prob"]


def config_with_queue_model(config, queue_model):
    values = config.__dict__.copy()
    values["queue_model"] = queue_model
    return BacktestConfig(**values)


def order_is_active(order, hbtpkg):
    return order is not None and int(order.status) != hbtpkg.FILLED and float(order.leaves_qty) > 0


def run_passive_bid_ask_strategy(config, hbtpkg, queue_model, qty=1.0, step_ns=1_000_000_000,
                                 max_steps=3_600, response_timeout_ns=10_000_000):
    queue_config = config_with_queue_model(config, queue_model)
    hbt = build_backtest(queue_config, hbtpkg)
    asset_no = 0
    rows = []
    filled_order_ids = set()

    try:
        wait_for_bbo(hbt, asset_no)
        depth = hbt.depth(asset_no)
        orders = [
            {"side": "buy", "order_id": 20_001, "price": float(depth.best_bid)},
            {"side": "sell", "order_id": 20_002, "price": float(depth.best_ask)},
        ]

        rows.append(record_order_state(
            hbt, hbtpkg, asset_no, queue_config, "initial_bbo", "passive_bid_ask", queue_model, qty=None,
        ))

        for spec in orders:
            rc = submit_limit_order(hbt, hbtpkg, asset_no, spec["order_id"], spec["side"], spec["price"], qty)
            response = hbt.wait_order_response(asset_no, spec["order_id"], response_timeout_ns)
            order = get_order(hbt, asset_no, spec["order_id"])
            if order is not None and float(order.exec_qty) > 0:
                raise RuntimeError(f"passive {spec['side']} order filled on accept: {spec}")
            rows.append(record_order_state(
                hbt, hbtpkg, asset_no, queue_config, "accepted", "passive_bid_ask", queue_model,
                side=spec["side"], order_id=spec["order_id"], px=spec["price"], qty=qty,
                rc=rc, response=response, step=0,
            ))

        for step in range(1, max_steps + 1):
            active_orders = []
            for spec in orders:
                order = get_order(hbt, asset_no, spec["order_id"])
                if order_is_active(order, hbtpkg):
                    active_orders.append(spec)

            if not active_orders:
                break

            if hbt.elapse(step_ns) != 0:
                break

            for spec in active_orders:
                order = get_order(hbt, asset_no, spec["order_id"])
                if order is None or spec["order_id"] in filled_order_ids:
                    continue
                if float(order.exec_qty) > 0 or int(order.status) == hbtpkg.FILLED:
                    rows.append(record_order_state(
                        hbt, hbtpkg, asset_no, queue_config, "filled", "passive_bid_ask", queue_model,
                        side=spec["side"], order_id=spec["order_id"], px=spec["price"], qty=qty, step=step,
                        is_fill=True,
                    ))
                    filled_order_ids.add(spec["order_id"])

        rows.append(record_order_state(
            hbt, hbtpkg, asset_no, queue_config, "final_state", "passive_bid_ask", queue_model, qty=None,
            step=step if "step" in locals() else 0,
        ))
        return pd.DataFrame(rows)
    finally:
        close_backtest(hbt)


def run_queue_model_comparison(config, hbtpkg, queue_models=QUEUE_MODELS, qty=1.0):
    output = pd.concat(
        [run_passive_bid_ask_strategy(config, hbtpkg, queue_model, qty=qty) for queue_model in queue_models],
        ignore_index=True,
    )
    fills = output[output["label"].eq("filled")].copy()
    fills = fills.sort_values(["side", "fill_ts", "queue_model"]).reset_index(drop=True)
    fills["fill_rank"] = fills.groupby("side").cumcount() + 1
    fills["fill_time_delta_ns"] = fills["fill_ts"] - fills.groupby("side")["fill_ts"].transform("min")
    return output, fills


In [ ]:
strategy2_output, strategy2_fill_comparison = run_queue_model_comparison(CONFIG, hbtpkg, qty=1.0)
strategy2_fill_comparison[
    [
        "queue_model", "side", "order_id", "price", "exec_price", "exec_qty",
        "exch_ts", "local_ts", "fill_ts", "exch_time", "local_time", "fill_time",
        "fill_time_delta_ns", "step", "position", "balance", "equity",
    ]
]